In [1]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.0 MB/s eta 0:00:00


In [2]:
import monai

print("MONAI Version:", monai.__version__)

MONAI Version: 1.6.0


In [4]:
!git clone https://github.com/ethicalcod/neuroBrain.git
%cd neuroBrain

Cloning into 'neuroBrain'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 86 (delta 33), reused 36 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 1.05 MiB | 12.10 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/neuroBrain


In [5]:
pwd

'/content/neuroBrain'

In [6]:
!python scripts/setup_project.py

NeuroBrain Project Setup
Installing tree...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Installing Python packages...
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1A2IU8Sgea1h3fYLpYtFb2v7NYdMjvEhU
From (redirected): https://drive.google.com/uc?id=1A2IU8Sgea1h3fYLpYtFb2v7NYdMjvEhU&confirm=t&uuid=083174c5-52c9-4376-88f5-cdd3869d1920
To: /content/neuroBrain/Task01_BrainTumour.tar
100% 7.61G/7.61G [01:33<00:00, 81.2MB/s]
Download complete.
Extracting archive...
/content/neuroBrain/scripts/setup_dataset.py:65: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modif

In [7]:
!python scripts/setup_dataset.py

 Dataset already exists.
/content/neuroBrain/Task01_BrainTumour


In [9]:
!python src/config.py

In [10]:
!python scripts/verify_project.py

NeuroBrain Project Verification
Project Root Exists!
Dataset Exists!
imagesTr Exists!
labelsTr Exists!
dataset.json Exists!
docs Exists!
figures Exists!
models Exists!
notebooks Exists!
results Doesn't exist!
scripts Exists!
src Exists!
Verification failed.


In [11]:
!ls

docs	 LICENSE  notebooks  requirements.txt  src
figures  models   README.md  scripts	       Task01_BrainTumour


In [12]:
import os
print(os.listdir())

['README.md', '.git', 'notebooks', 'requirements.txt', '.gitignore', 'src', 'Task01_BrainTumour', 'scripts', 'docs', 'figures', 'LICENSE', 'models']


In [13]:
!mkdir -p results

In [14]:
!python scripts/verify_project.py

NeuroBrain Project Verification
Project Root Exists!
Dataset Exists!
imagesTr Exists!
labelsTr Exists!
dataset.json Exists!
docs Exists!
figures Exists!
models Exists!
notebooks Exists!
results Exists!
scripts Exists!
src Exists!
 Everything looks good!


In [17]:
%%writefile scripts/setup_project.py

"""
Complete project setup.
"""

import shutil
import subprocess
from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.config import PROJECT_ROOT


print("NeuroBrain Project Setup")

# Create required directories

required_dirs = [
    "docs",
    "figures",
    "models",
    "notebooks",
    "results",
    "scripts",
    "src",
]

print("Checking project structure...")

for folder in required_dirs:
    path = PROJECT_ROOT / folder
    path.mkdir(parents=True, exist_ok=True)
    print(f"{folder} Exists!")


# Install tree (if needed)

if shutil.which("tree") is None:
    print("\nInstalling tree...")
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    subprocess.run(
        ["apt-get", "install", "-y", "tree"],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )


# Install Python requirements

requirements = PROJECT_ROOT / "requirements.txt"

if requirements.exists():
    print("Installing Python packages...")
    subprocess.run(
        ["pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

# Dataset setup

subprocess.run(
    [sys.executable, "scripts/setup_dataset.py"],
    check=True,
)

# Verification

subprocess.run(
    [sys.executable, "scripts/verify_project.py"],
    check=True,
)

# Show structure

print("\nProject structure:\n")

subprocess.run(
    ["tree", "-L", "2"],
    check=True,
)

print("\nSetup completed successfully.")


Overwriting scripts/setup_project.py


In [18]:
%%writefile scripts/verify_project.py
"""
Verify NeuroBrain project structure.
"""
from pathlib import Path
import sys

PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import *

checks = {
    "Project Root": PROJECT_ROOT,
    "Dataset": DATASET_ROOT,
    "imagesTr": IMAGES_PATH,
    "labelsTr": LABELS_PATH,
    "dataset.json": DATASET_JSON,
    "docs": DOCS_DIR,
    "figures": FIGURES_DIR,
    "models": MODELS_DIR,
    "notebooks": NOTEBOOKS_DIR,
    "results": RESULTS_DIR,
    "scripts": SCRIPTS_DIR,
    "src": SRC_DIR,
}

print("NeuroBrain Project Verification")

failed = False

for name, path in checks.items():
    if path.exists():
        print(f"{name} Exists!")
    else:
        print(f"{name} Doesn't exist!")
        failed = True

if failed:
    print("Verification failed.")
else:
    print(" Everything looks good!")

Overwriting scripts/verify_project.py


In [19]:
%%writefile src/transforms.py

"""
MONAI transform pipelines for NeuroBrain.
"""

from monai.transforms import (
    Compose,
    EnsureTyped,
)


def get_train_transforms():
    """
    Training transform pipeline.
    """

    return Compose(
        [
            EnsureTyped(
                keys=["image", "label"],
            ),
        ]
    )


def get_val_transforms():
    """
    Validation transform pipeline.
    """

    return Compose(
        [
            EnsureTyped(
                keys=["image", "label"],
            ),
        ]
    )


Writing src/transforms.py


In [20]:
import importlib
import src.transforms

importlib.reload(src.transforms)

<module 'src.transforms' from '/content/neuroBrain/src/transforms.py'>

In [21]:
from src.transforms import (
    get_train_transforms,
    get_val_transforms,
)

train_transform = get_train_transforms()
val_transform = get_val_transforms()

print(train_transform)
print()
print(val_transform)

In [23]:
for t in train_transform.transforms:
    print(t)

In [24]:
%%writefile src/loaders.py

"""
DataLoader utilities for NeuroBrain.
"""

import torch
from torch.utils.data import DataLoader, random_split

from src.config import RANDOM_SEED, BATCH_SIZE
from src.dataloader import BrainTumourDataset
from src.transforms import get_train_transforms


def create_dataloaders():
    """
    Create training and validation DataLoaders.
    """

    dataset = BrainTumourDataset(
        transform=get_train_transforms()
    )

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    generator = torch.Generator().manual_seed(RANDOM_SEED)

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=generator,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    return train_loader, val_loader

Overwriting src/loaders.py


In [25]:
import importlib
import src.loaders

importlib.reload(src.loaders)

train_loader, val_loader = src.loaders.create_dataloaders()

batch = next(iter(train_loader))

print("Image shape :", batch["image"].shape)
print("Label shape :", batch["label"].shape)

print("Image dtype :", batch["image"].dtype)
print("Label dtype :", batch["label"].dtype)

Images : 484
Labels : 484
Dataset verification passed.
Image shape : torch.Size([2, 4, 240, 240, 155])
Label shape : torch.Size([2, 240, 240, 155])
Image dtype : torch.float32
Label dtype : torch.int64


In [26]:
!git config --global user.name Shashi
!git config --global user.email 173938981+ethicalcod@users.noreply.github.com

In [28]:
!git add .
!git commit -m "Initial MONAI integration with a transform pipeline"

[main 5b66d66] Initial MONAI integration with a transform pipeline
 3 files changed, 76 insertions(+), 10 deletions(-)
 create mode 100644 src/transforms.py


In [30]:
!git push origin main

Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 1.21 KiB | 1.21 MiB/s, done.
Total 7 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/ethicalcod/neuroBrain.git
   aa1392f..5b66d66  main -> main
